# Retrain the thyroid image model on TN5000 (official split)

Trains EfficientNet-B0 to classify thyroid ultrasound images as **benign** or **malignant**.

* Train on TN5000's official **train** split (3,500 images), pick the best epoch on **val** (500), score the untouched **test** split (1,000) once.
* Class-weighted loss (the data is ~70% malignant) and ImageNet normalisation.
* **Independent test:** the trained model is also scored on the TN3K test set, from a different hospital and study, to show how it generalises beyond TN5000.
* Outputs: `cancer_efficientnet.pth`, `cancer_efficientnet.json` (preprocessing + metrics), `metrics.json`, `roc.png`, `confusion_matrix.png`, `history.csv`.

**Before you run:** `Runtime → Change runtime type → T4 GPU`. The notebook downloads TN5000 (~200 MB) from figshare itself and checks the file; Google Drive is only used to save the trained model.

Dataset (CC BY): Zhang et al., *TN5000: An Ultrasound Image Dataset for Thyroid Nodule Detection and Classification*, Scientific Data 12, 1437 (2025).

In [ ]:
# 1. Get TN5000 (about 1 minute). Tries, in order:
#    a) TN5000_forReview.zip already in your Google Drive (My Drive)
#    b) direct download from figshare
#    c) upload from your computer (a file picker appears below)
import os, shutil, subprocess, zipfile
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/TN5000.zip'
DRIVE_ZIP = '/content/drive/MyDrive/TN5000_forReview.zip'
URL = 'https://figshare.com/ndownloader/files/52597376?private_link=cb6a67f17c04b29e7edd'

def looks_ok(path):
    try:
        with zipfile.ZipFile(path) as z:
            return any(n.endswith('ImageSets/Main/test.txt') for n in z.namelist())
    except Exception:
        return False

if not looks_ok(ZIP) and os.path.exists(DRIVE_ZIP):
    print('Copying from Google Drive...'); shutil.copy(DRIVE_ZIP, ZIP)
if not looks_ok(ZIP):
    print('Downloading from figshare...')
    subprocess.run(['wget', '-q', '--user-agent=Mozilla/5.0', '-O', ZIP, URL])
if not looks_ok(ZIP):
    print('figshare blocked the download. Choose TN5000_forReview.zip from your computer '
          '(Downloads folder); uploading ~200 MB takes a few minutes.')
    from google.colab import files
    up = files.upload()
    shutil.move(next(iter(up)), ZIP)

assert looks_ok(ZIP), 'Still not a valid TN5000 zip. Put TN5000_forReview.zip in My Drive and re-run this cell.'
print(f'TN5000 ready: {os.path.getsize(ZIP)/1e6:.0f} MB')

In [ ]:
# 2. Get the project code (safe to re-run)
%cd /content
!rm -rf /content/thyroid-ml-system
!git clone -q https://github.com/lonewolf1621/thyroid-ml-system /content/thyroid-ml-system
%cd /content/thyroid-ml-system
!git log --oneline -1
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (enable a GPU runtime!)')

In [ ]:
# 3. Train (about 10-15 minutes on a T4 GPU)
!python src/train_image_model.py --data /content/TN5000.zip --out /content/outputs --epochs 15 --batch-size 32

In [ ]:
# 4. Results on the held-out test set
import json
from IPython.display import Image, display
m = json.load(open('/content/outputs/metrics.json'))['test']
for k in ['accuracy','balanced_accuracy','sensitivity','specificity','precision','f1','auc','majority_class_baseline']:
    print(f'{k:>24}: {m[k]:.3f}')
print('confusion matrix:', m['confusion_matrix'])
display(Image('/content/outputs/roc.png'))
display(Image('/content/outputs/confusion_matrix.png'))

## Independent test on TN3K

TN3K (Gong et al., ISBI 2021 / MICCAI 2022) was collected at a different hospital, with different scanners, by a different team. The model never sees it during training, so this is the honest answer to *"how would it do on new images?"*. Expect a lower score than on TN5000: that gap is the point of the test.

The data comes from the authors' public Google Drive folder ([ACL repository](https://github.com/chenghui-666/ACL)). If the download fails with a Drive quota error, wait an hour and re-run, or download `datasets.rar` from that folder manually into `/content/tn3k`.

In [ ]:
# 5. Download TN3K and score the trained model on its test set (about 2-3 minutes)
!pip -q install gdown
!apt-get -qq install -y unrar > /dev/null
!mkdir -p /content/tn3k
!cd /content/tn3k && gdown -q --folder https://drive.google.com/drive/folders/1YtZGTb-AOmVk_uRZGWsT-pjK8hOAUix6
!cd /content/tn3k && find . -name '*.rar' -exec unrar x -o+ -inul {} \;
!python src/evaluate_external.py --model /content/outputs/cancer_efficientnet.pth --tn3k /content/tn3k

In [ ]:
# 6. Compare: TN5000 test (same source as training) vs TN3K (independent)
import json
from IPython.display import Image, display
card = json.load(open('/content/outputs/cancer_efficientnet.json'))['metrics']
rows = [('TN5000 test', card['test']), ('TN3K external', card.get('external_tn3k'))]
print(f"{'':16}{'acc':>7}{'bal.acc':>9}{'sens':>7}{'spec':>7}{'AUC':>7}{'n':>6}")
for name, m in rows:
    if m: print(f"{name:16}{m['accuracy']:7.3f}{m['balanced_accuracy']:9.3f}{m['sensitivity']:7.3f}{m['specificity']:7.3f}{m['auc']:7.3f}{m['n']:6d}")
display(Image('/content/outputs/external_tn3k_roc.png'))

In [ ]:
# 7. Save everything to Google Drive (folder: My Drive/thyroid_model_v2)
!mkdir -p /content/drive/MyDrive/thyroid_model_v2
!cp /content/outputs/* /content/drive/MyDrive/thyroid_model_v2/
!ls -lh /content/drive/MyDrive/thyroid_model_v2